在开始该项目之前，您需要先安装 PyTorch；您可以在此网页上找到所需的安装命令：https://pytorch.org/get-started/locally/ 。具体命令取决于您的编程语言和计算平台。例如，我使用的是这条命令：`pip install torch torchvision`，然后按下 Shift+Enter 即可

1.先将所需要的库导入到项目中

In [32]:
import torch
import torchvision
from torch.utils.data import DataLoader #Dataloader用于加载数据集，把数据从dataset里批量拿过来，同时可以利用cpu或gpu多核处理提高速度

2.定义所要使用的参数

In [33]:
n_epochs=1 #epochs:将整个训练集完整训练一遍的次数
batch_size_train=64 #batch size:每批训练所用的数据量
batch_size_test=1000
learning_rate=0.01 #learning rate:梯度下降时所更新的步长
momentum=0.5 #momentum:参考之前几次梯度下降的方向，使之后下降更平滑
log_interval=10 #日志输入间隔，每10个batch打印一次
random_seed=1 #过程随机生成后 固定
torch.manual_seed(random_seed)

3.导入数据集

In [34]:
train_loader=torch.utils.data.DataLoader(
    torchvision.datasets.MNIST('D:/Data/',train=True,download=True,
                               transform=torchvision.transforms.Compose([
                               torchvision.transforms.ToTensor(),#将图片转换为Tensor，也就是转化成模型可以理解的数据
                               torchvision.transforms.Normalize(
                               (0.1307,),(0.3081,)
                               )#将数据的值以0为中心，让梯度下降更稳且快
                               ])),
    batch_size=batch_size_train, shuffle=True)
    
test_loader=torch.utils.data.DataLoader(
    torchvision.datasets.MNIST('D:/Data/',train=False,download=True,
                              transform=torchvision.transforms.Compose([
                              torchvision.transforms.ToTensor(),
                              torchvision.transforms.Normalize(
                              (0.1307,),(0.3081,)
                              )]
                              )),
    batch_size=batch_size_test,shuffle=True)

4.构建神经网络

In [35]:
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

class Net(nn.Module):
    def __init__(self):
        super(Net,self).__init__()
        self.conv1=nn.Conv2d(1,10,kernel_size=5)#卷积层，第一层输入一个数字，也就是一个通道，输出10个
        self.conv2=nn.Conv2d(10,20,kernel_size=5)#卷积层，接第一层的10个通道，输出20个通道
        self.conv2_drop=nn.Dropout2d()#池化层，随机遮挡部分特征图，防止过拟合
        self.fc1=nn.Linear(320,50)#全连接层，将提取出的图像特征映射到神经元上
        self.fc2=nn.Linear(50,10)#全连接层，输出10个神经元对应数字0-9
    def forward(self,x):
        x=F.relu(F.max_pool2d(self.conv1(x),2))#执行第一次卷积->pool进行2*2池化减小图像->Relu函数激活
        x=F.relu(F.max_pool2d(self.conv2_drop(self.conv2(x)),2))#第二次卷积后随机遮挡
        x=x.view(-1,320)#展平，因为前面的卷积和池化输出都是多维向量，后面的全连接层只能接受一维向量
        x=F.relu(self.fc1(x))
        x=F.dropout(x,training=self.training)#训练时随机关掉部分神经元
        x=self.fc2(x)
        return F.log_softmax(x,dim=1)#把得分变成标准（0-1）的对数概率分布并返回

5.初始化网络和优化器

In [36]:
network = Net()
optimizer = optim.SGD(network.parameters(), lr=learning_rate, momentum=momentum)#gradient descent

6.定义一些数据方便之后记录训练过程

In [37]:
train_losses = []
train_counter = []
test_losses = []
test_counter = [i*len(train_loader.dataset) for i in range(n_epochs + 1)]

7.训练模型

In [38]:
def train(epoch):
    network.train()#训练模式
    for batch_idx, (data, target) in enumerate(train_loader):#提取出trainloader中的batchidx这批数据的编号，data张量和target真实标签进行循环
        optimizer.zero_grad()#将上一批算出来的梯度清零，在新起点再开始下一批训练
        output = network(data)#向前传播，把data输入网络
        loss = F.nll_loss(output, target)#计算损失
        loss.backward()#反向传播，自动计算出loss对每一个参数的梯度，即找出每个输出出来的参数所需要承担错误的责任有多少
        optimizer.step()#更新参数根据上一步算出来的梯度，再次微调一遍
        if batch_idx % log_interval == 0:#控制打印频率
            print('Train Epoch: {} [{}/{} ({:.0f}%)]\tLoss: {:.6f}'.format(epoch, batch_idx * len(data),
                                                                           len(train_loader.dataset),
                                                                           100. * batch_idx / len(train_loader),
                                                                           loss.item()))
            train_losses.append(loss.item())#item（）将损失值转化为普通数字存入train_losses列表里
            train_counter.append((batch_idx * 64) + ((epoch - 1) * len(train_loader.dataset)))#记录训练了多少样本
            torch.save(network.state_dict(), './model.pth')#保存参数到.pth文件里
            torch.save(optimizer.state_dict(), './optimizer.pth')#同上
 
def test():
    network.eval()#测试模式
    test_loss = 0
    correct = 0
    with torch.no_grad():#关闭梯度下降
        for data, target in test_loader:#在测试集中提取出data和target
            output = network(data)#把输出值存在output里
            test_loss += F.nll_loss(output, target, reduction='sum').item()#将损失值累加起来
            pred = output.data.max(1, keepdim=True)[1]#找出概率最大的数字和索引
            correct += pred.eq(target.data.view_as(pred)).sum()#统计匹配正确的数量
    test_loss /= len(test_loader.dataset)#计算平均损失
    test_losses.append(test_loss)#记录到test_losses列表里
    print('\nTest set: Avg. loss: {:.4f}, Accuracy: {}/{} ({:.0f}%)\n'.format(
        test_loss, correct, len(test_loader.dataset),
        100. * correct / len(test_loader.dataset)))
 
 
train(n_epochs)
test()

Train Epoch: 1 [0/60000 (0%)]	Loss: 2.329236
Train Epoch: 1 [640/60000 (1%)]	Loss: 2.331165
Train Epoch: 1 [1280/60000 (2%)]	Loss: 2.295031
Train Epoch: 1 [1920/60000 (3%)]	Loss: 2.275138
Train Epoch: 1 [2560/60000 (4%)]	Loss: 2.283847
Train Epoch: 1 [3200/60000 (5%)]	Loss: 2.228793
Train Epoch: 1 [3840/60000 (6%)]	Loss: 2.198277
Train Epoch: 1 [4480/60000 (7%)]	Loss: 2.154135
Train Epoch: 1 [5120/60000 (9%)]	Loss: 2.065568
Train Epoch: 1 [5760/60000 (10%)]	Loss: 1.939048
Train Epoch: 1 [6400/60000 (11%)]	Loss: 1.946784
Train Epoch: 1 [7040/60000 (12%)]	Loss: 1.909579
Train Epoch: 1 [7680/60000 (13%)]	Loss: 1.701835
Train Epoch: 1 [8320/60000 (14%)]	Loss: 1.696330
Train Epoch: 1 [8960/60000 (15%)]	Loss: 1.610522
Train Epoch: 1 [9600/60000 (16%)]	Loss: 1.593356
Train Epoch: 1 [10240/60000 (17%)]	Loss: 1.553732
Train Epoch: 1 [10880/60000 (18%)]	Loss: 1.521820
Train Epoch: 1 [11520/60000 (19%)]	Loss: 1.125519
Train Epoch: 1 [12160/60000 (20%)]	Loss: 1.070072
Train Epoch: 1 [12800/60000 (